# 06. Sensitivity of the Classification to the Discretization Rule (supplementary analysis; not reported in the final manuscript)

> Reproduction notebook for *Addressing the Psychological Small-Sample Problem with Multimodal Large Language Models: Evidence from Multiple Moral Emotion Recognition Study*. Section numbers refer to the manuscript appendices. Every notebook reads only the committed files in `../final/` and ends with a check against the frozen outputs in `../final/reproducibility/`. Run the notebooks in numerical order; `00` writes the label tables that the others reuse.

Implements the discretization-sensitivity analysis (formerly Methods 2.4.6 / Appendix E.6; removed from the final manuscript) for the ten MLLM configurations (Exp 1–9 and 16). The four-class labels are produced by cutting each continuous score at a threshold; this notebook quantifies how much of each configuration's accuracy depends on that rule. For every MLLM configuration it recomputes four-class accuracy under the sample-mean rule (the reported rule), the scale midpoint (5.5), the sample median, and the best pair of fixed half-point thresholds found by grid search with knowledge of the labels (a label-informed, in-sample upper bound, not an estimate of out-of-sample accuracy; column `acc_oracle`). It also records the proportion of participants whose class changes between the mean and midpoint rules, the number of scores at the two integers that bracket the mean threshold, and the threshold-free macro-AUC.

In [1]:
import os, sys, json
import numpy as np, pandas as pd
sys.path.insert(0, os.path.abspath("."))
import common as C
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 40)
rng = np.random.default_rng(C.SEED)
print("final data folder:", os.path.relpath(C.FINAL))

final data folder: ../final


## 1. Load scores

In [2]:
ALL = pd.read_csv(os.path.join(C.OUT, "labels_all.csv"))
LAB = {e: g.drop(columns="exp").reset_index(drop=True) for e, g in ALL.groupby("exp") if e in C.MLLM_EXPS}

## 2. Accuracy under alternative rules

In [3]:
rows = []
for e in C.MLLM_EXPS:
    d = LAB[e]; ts, ti = d.symp.mean(), d.insp.mean(); t = d.true.values
    acc = lambda a, b: (C.labels_with_thresholds(d.symp, d.insp, a, b) == t).mean()
    grid = np.arange(0.5, 10, 0.5); best = max(((acc(a, b), a, b) for a in grid for b in grid))
    changed = (C.labels_with_thresholds(d.symp, d.insp, ts, ti) != C.labels_with_thresholds(d.symp, d.insp, 5.5, 5.5)).mean()
    ks, ki = int(np.floor(ts)), int(np.floor(ti))
    sa, ia, _ = C.axis_accuracy(t, d.pred)
    rows.append(dict(exp=e, n=len(d), thr_s=ts, thr_i=ti, acc_mean=acc(ts, ti), acc_mid=acc(5.5, 5.5), acc_median=acc(d.symp.median(), d.insp.median()),
                     acc_oracle=best[0], thr_or_s=best[1], thr_or_i=best[2], changed_mean_vs_mid=changed, symp_axis=sa, insp_axis=ia,
                     extreme_share=0.5 * np.mean((d.symp <= 2) | (d.symp >= 8)) + 0.5 * np.mean((d.insp <= 2) | (d.insp >= 8)),
                     adj_s=int(((d.symp == ks) | (d.symp == ks + 1)).sum()), adj_i=int(((d.insp == ki) | (d.insp == ki + 1)).sum()),
                     sd_s=d.symp.std(), sd_i=d.insp.std(), macro_auc=C.macro_auc(d)))
TS = pd.DataFrame(rows); TS.to_csv(os.path.join(C.OUT, "threshold_sensitivity.csv"), index=False)
TS[["exp", "n", "thr_s", "thr_i", "adj_s", "adj_i", "acc_mean", "acc_mid", "acc_median", "acc_oracle", "changed_mean_vs_mid", "macro_auc"]].round(3)

Out[3]: 
   exp    n  thr_s  thr_i  adj_s  adj_i  acc_mean  acc_mid  acc_median  acc_oracle  changed_mean_vs_mid  macro_auc
0    1   96  1.344  1.052     88     94     0.333    0.260       0.333       0.333                0.188      0.572
1    2  100  2.390  1.850     21     82     0.490    0.400       0.530       0.530                0.370      0.702
2    3  100  3.690  3.540      5      6     0.630    0.630       0.630       0.660                0.060      0.823
3    4  100  5.020  4.860      5      3     0.690    0.700       0.710       0.740                0.010      0.902
4    5  100  3.510  3.850      7      8     0.660    0.660       0.680       0.700                0.140      0.838
5    6  100  3.950  3.940      5      7     0.700    0.660       0.760       0.760                0.080      0.873
6    7  100  3.780  3.720      4      6     0.660    0.660       0.600       0.670                0.040      0.795
7    8  100  4.130  3.580      3      5     0.660    0.640       0.710 

## 3. Score distributions behind the pattern

Configurations with language input produce bimodal scores (most values at 1–2 or 8–9), so few participants sit near any plausible cut; the two configurations without language pile up at the floor, where the mean falls between adjacent integers.

In [4]:
for e in (1, 2, 4, 9):
    d = LAB[e]; print(f"Exp {e} Sympathy:", d.symp.value_counts().sort_index().to_dict()); print(f"      Inspiration:", d.insp.value_counts().sort_index().to_dict())

Exp 1 Sympathy: {1.0: 80, 2.0: 8, 3.0: 3, 4.0: 3, 5.0: 1, 7.0: 1}
      Inspiration: {1.0: 93, 2.0: 1, 3.0: 2}
Exp 2 Sympathy: {1.0: 56, 2.0: 17, 3.0: 4, 4.0: 6, 5.0: 2, 6.0: 5, 7.0: 7, 8.0: 3}
      Inspiration: {1.0: 68, 2.0: 14, 3.0: 5, 4.0: 4, 5.0: 3, 6.0: 2, 7.0: 1, 8.0: 3}
Exp 4 Sympathy: {1.0: 34, 2.0: 7, 3.0: 1, 4.0: 3, 5.0: 1, 6.0: 4, 7.0: 14, 8.0: 12, 9.0: 24}
      Inspiration: {1.0: 31, 2.0: 9, 3.0: 8, 4.0: 2, 5.0: 1, 6.0: 2, 7.0: 8, 8.0: 19, 9.0: 20}
Exp 9 Sympathy: {1.0: 45, 2.0: 7, 3.0: 1, 4.0: 3, 5.0: 4, 7.0: 7, 8.0: 17, 8.5: 1, 9.0: 13, 10.0: 1}
      Inspiration: {1.0: 47, 2.0: 6, 4.0: 3, 5.0: 4, 6.0: 4, 7.0: 9, 8.0: 13, 9.0: 13}


## 4. Verification against the frozen outputs

In [5]:
print(C.compare(TS, C.load_frozen("threshold_sensitivity_frozen.csv"), ["exp"], ["n", "thr_s", "thr_i", "acc_mean", "acc_mid", "acc_median", "acc_oracle", "changed_mean_vs_mid", "symp_axis", "insp_axis", "extreme_share", "adj_s", "adj_i", "macro_auc"], atol=0.001).to_string(index=False))

             column  n_compared  max_abs_diff  within_tolerance
                  n          10  0.000000e+00              True
              thr_s          10  0.000000e+00              True
              thr_i          10  0.000000e+00              True
           acc_mean          10  0.000000e+00              True
            acc_mid          10  0.000000e+00              True
         acc_median          10  0.000000e+00              True
         acc_oracle          10  0.000000e+00              True
changed_mean_vs_mid          10  0.000000e+00              True
          symp_axis          10  0.000000e+00              True
          insp_axis          10  0.000000e+00              True
      extreme_share          10  0.000000e+00              True
              adj_s          10  0.000000e+00              True
              adj_i          10  0.000000e+00              True
          macro_auc          10  1.110223e-16              True
